# TabFM PyTorch Quickstart + Benchmark

Loads TabFM PyTorch checkpoints and runs the multi-dataset benchmark defined in
[`src/tabfm_benchmark`](../src/tabfm_benchmark), storing results under `artifacts/`.

New to TabFM? Start with [`docs/00-overview.md`](../docs/00-overview.md) and
[`docs/03-first-run.md`](../docs/03-first-run.md) first — this notebook assumes you already know what
"default" vs. "ensemble preset" means (explained in
[`docs/04-core-concepts.md §5`](../docs/04-core-concepts.md#5-the-ensemble-preset-what-it-actually-turns-on)).

> **License reminder (read this first):** TabFM's released checkpoint weights are distributed under the
> **TabFM Non-Commercial License v1.0** — research/evaluation only. See
> [`docs/09-faq.md#licensing`](../docs/09-faq.md#licensing) before any commercial use.

In [ ]:
from pathlib import Path

from loguru import logger
import torch

# This repo's convention: use a GPU only if it has at least 12 GiB of VRAM (TabFM's
# checkpoints alone use several GB); otherwise fall back to CPU. See
# docs/08-troubleshooting.md#cuda-out-of-memory for why this threshold exists.
DEVICE = "cpu"
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    DEVICE = "cuda" if gpu_mem_gb >= 12.0 else "cpu"
    logger.info("Detected GPU: {} ({:.1f} GiB)", torch.cuda.get_device_name(0), gpu_mem_gb)
    if DEVICE == "cpu":
        logger.warning("GPU has <12 GiB VRAM — using CPU for a stable run.")
else:
    logger.info("No CUDA GPU detected — using CPU.")

logger.info("Using device: {}", DEVICE)

## 1. Load TabFM classification and regression models

In [ ]:
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

clf_model = tabfm_v1_0_0.load(model_type="classification", device=DEVICE)
reg_model = tabfm_v1_0_0.load(model_type="regression", device=DEVICE)

logger.info("Loaded model types: {}, {}", type(clf_model).__name__, type(reg_model).__name__)

## 2. Run multi-dataset benchmark

In [ ]:
import polars as pl

from tabfm_benchmark import render_summary, run_benchmark, save_results

results_df = run_benchmark(
    device=DEVICE,
    seed=42,
    n_estimators=32,
    use_ensemble_preset=True,
)
summary_df = render_summary(results_df)

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

save_results(results_df, artifacts_dir / "tabfm_benchmark_results.parquet")
save_results(summary_df, artifacts_dir / "tabfm_benchmark_summary.csv")

logger.info("Saved artifacts under {}", artifacts_dir.resolve())
results_df

## 3. Plot ranking views

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

pdf = results_df.filter(pl.col("status") == "ok").to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cls_df = pdf[pdf["task_type"] == "classification"].dropna(subset=["accuracy"])
if not cls_df.empty:
    cls_df = cls_df.sort_values("accuracy", ascending=False)
    sns.barplot(data=cls_df, x="accuracy", y="dataset", ax=axes[0], color="steelblue")
    axes[0].set_title("Classification Accuracy")
else:
    axes[0].set_title("Classification Accuracy (no successful runs)")

reg_df = pdf[pdf["task_type"] == "regression"].dropna(subset=["rmse"])
if not reg_df.empty:
    reg_df = reg_df.sort_values("rmse", ascending=True)
    sns.barplot(data=reg_df, x="rmse", y="dataset", ax=axes[1], color="darkorange")
    axes[1].set_title("Regression RMSE (lower is better)")
else:
    axes[1].set_title("Regression RMSE (no successful runs)")

plt.tight_layout()
plt.show()

## Next steps

See [`docs/07-evaluation.md`](../docs/07-evaluation.md) for how to read these results against a
baseline, and [`docs/10-next-steps.md`](../docs/10-next-steps.md) for where to go from here.